# Figure 03 final source rebuild

Run top-to-bottom. The notebook recomputes its figure from the H5AD, paired OME-TIFF files, and OpenRouter API key.


## 1. Configure the three source inputs


In [ ]:
import os
from pathlib import Path
from IPython.display import display


def find_repository_root() -> Path:
    configured = os.environ.get("SOURCE_REBUILD_REPO_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "source_rebuild_scripts").is_dir():
            raise RuntimeError(f"SOURCE_REBUILD_REPO_ROOT is not the repository root: {root}")
        return root
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "source_rebuild_scripts").is_dir():
            return candidate
    raise RuntimeError("Run from the repository checkout or set SOURCE_REBUILD_REPO_ROOT")


REPO_ROOT = find_repository_root()

# The only source inputs are the H5AD, paired OME-TIFF directories, and an API key.
# Edit paths here or set the same variables before launching Jupyter.
os.environ.setdefault("SOURCE_REBUILD_H5AD_PATH", str(REPO_ROOT / "data" / "20251007_cleaned_trainingdata_yang.h5ad"))
os.environ.setdefault("SOURCE_REBUILD_TIFF_ROOT", str(REPO_ROOT / "data" / "tiff"))
os.environ.setdefault("SOURCE_REBUILD_LLM_MODE", "live")
os.environ.setdefault("SOURCE_REBUILD_LLM_OUTPUT_ROOT", str(REPO_ROOT / "outputs" / "source_rebuilt" / "raw_llm"))
os.environ.setdefault("SOURCE_REBUILD_OUTPUT_ROOT", str(REPO_ROOT / "outputs" / "source_rebuilt"))

# Leave this literal placeholder in the notebook. Supply the real key through the shell/Jupyter environment.
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "REPLACE_WITH_OPENROUTER_API_KEY")
if os.environ["SOURCE_REBUILD_LLM_MODE"].strip().lower() == "live" and OPENROUTER_API_KEY == "REPLACE_WITH_OPENROUTER_API_KEY":
    raise RuntimeError("Set OPENROUTER_API_KEY in the environment before running live LLM annotation")


## 2. Load source data and define the reproducible analysis


In [ ]:
# Source loading, TIFF validation, clustering, and LLM annotation implementation.

#!/usr/bin/env python3.12
"""Source-only runtime for the canonical final figure notebooks.

The runtime reads only the Yang H5AD, explicitly paired OME-TIFF files, and
raw LLM API bundles. Cluster assignments, embeddings, metrics, and model labels
are held in memory for one notebook process. Scratch material created by a
clustering backend is disposable output and is never a later input.
"""

import hashlib
import json
import os
import re
import shutil
import sys
import tempfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal, Mapping
from urllib import request as urllib_request

import numpy as np
import pandas as pd


ROOT = REPO_ROOT
V2_SRC = ROOT / "LLM-Spatial-omics-Clustering" / "src"
if str(V2_SRC) not in sys.path:
    sys.path.insert(0, str(V2_SRC))

DEFAULT_H5AD = ROOT / "data" / "20251007_cleaned_trainingdata_yang.h5ad"
DEFAULT_TIFF_ROOT = ROOT / "data" / "tiff"
DEFAULT_LLM_ROOT = ROOT / "data" / "llm_api"
PLACEHOLDER_API_KEY = "REPLACE_WITH_OPENROUTER_API_KEY"

COHORT_FILE_IDS = (
    "2e65eeef2dd18bee2a0baf1cec6d35a1",
    "5318485b16983482401c3be24b6c42ad",
    "63d000170e475af142f6e8673de5eb0f",
    "768b7adb649959b6b7b8741c282677eef",
    "76d3efd17b6fc83aaac13e961824c5ae",
    "8da8f27977d946b8c912d42c8827b55c",
    "ae422532f260b3d6fc662aae69b05d33",
    "dceadbb36871071f30c308ca091fbdc8",
)

METHODS = ("leiden", "flowsom", "spatialsort", "pixie_tiff")
MODELS = ("gpt", "claude", "gemini", "deepseek")
MODEL_IDS = {
    "gpt": "openai/gpt-5.6-sol",
    "claude": "anthropic/claude-opus-5",
    "gemini": "google/gemini-3.1-pro-preview",
    "deepseek": "deepseek/deepseek-v4-pro",
}
METHOD_LABELS = {"leiden": "Leiden", "flowsom": "FlowSOM", "spatialsort": "SpatialSort", "pixie_tiff": "TIFF PIXIE"}
MODEL_LABELS = {"gpt": "GPT", "claude": "Claude", "gemini": "Gemini", "deepseek": "DeepSeek"}

TRUTH_MAP = {
    "Epithelial": "Enterocyte",
    "CD66+ Epithelial": "CD66+ Enterocyte",
    "MUC1+ Epithelial": "MUC1+ Enterocyte",
    "CD57+ Epithelial": "Enterocyte",
    "ITLN+ Epithelial": "Goblet",
    "TA": "Cycling TA",
    "CD4+ T": "CD4+ T cell",
    "Stromal": "Stroma",
    "PDPN+ Stromal": "Lymphatic",
    "CD36 high Endothelial": "Endothelial",
    "CD36 low Endothelial": "Endothelial",
    "CD49a+ Smooth muscle": "Smooth muscle",
    "Paneth": "Neuroendocrine",
    "M1 Macrophage": "M2 Macrophage",
    "NK": "CD7+ Immune",
}
ONTOLOGY = (
    "B", "CD4+ T cell", "CD66+ Enterocyte", "CD7+ Immune", "CD8+ T", "Cycling TA", "DC",
    "Endothelial", "Enterocyte", "Goblet", "ICC", "Lymphatic", "M2 Macrophage",
    "MUC1+ Enterocyte", "Nerve", "Neuroendocrine", "Neutrophil", "Noise", "Plasma",
    "Smooth muscle", "Stroma",
)


class SourceRebuildError(RuntimeError):
    """Raised when a source-only rebuild cannot satisfy its input contract."""


@dataclass(frozen=True)
class TiffPair:
    file_id: str
    expression: Path
    mask: Path


@dataclass(frozen=True)
class SourceInputs:
    h5ad: Path
    tiff_root: Path
    llm_data_root: Path
    llm_mode: Literal["cached", "live"] = "cached"
    api_keys: Mapping[str, str] = field(default_factory=lambda: {"openrouter": PLACEHOLDER_API_KEY})
    live_output_root: Path | None = None

    @classmethod
    def from_environment(cls) -> "SourceInputs":
        mode = os.environ.get("SOURCE_REBUILD_LLM_MODE", "live").strip().lower()
        if mode not in {"cached", "live"}:
            raise SourceRebuildError("SOURCE_REBUILD_LLM_MODE must be 'cached' or 'live'")
        return cls(
            h5ad=Path(os.environ.get("SOURCE_REBUILD_H5AD_PATH", str(DEFAULT_H5AD))).expanduser().resolve(),
            tiff_root=Path(os.environ.get("SOURCE_REBUILD_TIFF_ROOT", str(DEFAULT_TIFF_ROOT))).expanduser().resolve(),
            llm_data_root=Path(os.environ.get("SOURCE_REBUILD_LLM_DATA_ROOT", str(DEFAULT_LLM_ROOT))).expanduser().resolve(),
            llm_mode=mode,  # type: ignore[arg-type]
            api_keys={"openrouter": os.environ.get("OPENROUTER_API_KEY", PLACEHOLDER_API_KEY)},
            live_output_root=(Path(os.environ["SOURCE_REBUILD_LLM_OUTPUT_ROOT"]).expanduser().resolve() if os.environ.get("SOURCE_REBUILD_LLM_OUTPUT_ROOT") else None),
        )

    def tiff_pairs(self) -> tuple[TiffPair, ...]:
        return tuple(
            TiffPair(file_id, self.tiff_root / file_id / "reg001_expr.ome.tif", self.tiff_root / file_id / "reg001_mask.ome.tif")
            for file_id in COHORT_FILE_IDS
        )


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _require_file(path: Path, label: str) -> Path:
    path = path.expanduser().resolve()
    if not path.is_file():
        raise SourceRebuildError(f"{label} is missing: {path}")
    return path


def _canonical_cluster(value: object) -> str:
    text = str(value).strip()
    match = re.search(r"(?:cluster|target|group)[_:\- ]*([0-9]+)$", text, re.IGNORECASE)
    if match:
        return str(int(match.group(1)))
    return str(int(text)) if text.isdigit() else text


def _infer_method(path: Path, payload: Mapping[str, Any]) -> str | None:
    provenance = payload.get("provenance")
    values = [payload.get("call_id"), path.name]
    if isinstance(provenance, Mapping):
        values.append(provenance.get("request_metadata", {}).get("call_id"))
    text = " ".join(str(value).lower() for value in values if value)
    return next((method for method in METHODS if method in text), None)


def _infer_model(path: Path, payload: Mapping[str, Any]) -> str | None:
    provenance = payload.get("provenance")
    values = [payload.get("model_key"), path.name]
    if isinstance(provenance, Mapping):
        values.append(provenance.get("model_key"))
    text = " ".join(str(value).lower() for value in values if value)
    return next((model for model in MODELS if model in text), None)


def _annotation_pairs(payload: Mapping[str, Any]) -> list[tuple[str, str]]:
    annotations = payload.get("annotations")
    if isinstance(annotations, Mapping):
        items = annotations.items()
    elif isinstance(annotations, list):
        items = ((item.get("target_id"), item.get("label")) for item in annotations if isinstance(item, Mapping))
    else:
        items = ()
    pairs: list[tuple[str, str]] = []
    for target, value in items:
        if isinstance(value, Mapping):
            value = value.get("label") or value.get("predicted_label") or value.get("annotation")
        if target is None or value is None:
            continue
        label = str(value).strip()
        if label not in ONTOLOGY:
            raise SourceRebuildError(f"LLM response contains a label outside the ontology: {label!r}")
        pairs.append((_canonical_cluster(target), label))
    return pairs


def _json_files(root: Path, directory_name: str) -> list[Path]:
    if not root.is_dir():
        raise SourceRebuildError(f"LLM API data root is missing: {root}")
    return sorted(path for path in root.rglob("*.json") if directory_name in path.parts)


class SourceContext:
    """Lazy source data, clustering, annotation, and metric state."""

    def __init__(self, inputs: SourceInputs):
        self.inputs = inputs
        self._scratch = Path(tempfile.mkdtemp(prefix="cell_masks_final_source_rebuild_"))
        self._prepared = False
        self._cells: pd.DataFrame | None = None
        self._features: np.ndarray | None = None
        self._markers: tuple[str, ...] | None = None
        self._assignments: dict[str, pd.DataFrame] = {}
        self._predictions: dict[tuple[str, str], pd.DataFrame] = {}
        self._embedding: np.ndarray | None = None

    def close(self) -> None:
        shutil.rmtree(self._scratch, ignore_errors=True)

    def status(self) -> dict[str, object]:
        pairs = self.inputs.tiff_pairs()
        return {
            "runtime_boundary": "H5AD + paired OME-TIFF files + OpenRouter key (or cached raw LLM API bundles)",
            "h5ad": str(self.inputs.h5ad),
            "h5ad_exists": self.inputs.h5ad.is_file(),
            "tiff_root": str(self.inputs.tiff_root),
            "tiff_pair_count": len(pairs),
            "tiff_pairs_present": all(pair.expression.is_file() and pair.mask.is_file() for pair in pairs),
            "llm_data_root": str(self.inputs.llm_data_root),
            "llm_mode": self.inputs.llm_mode,
            "api_key_persisted": False,
            "prepared": self._prepared,
        }

    @property
    def cells(self) -> pd.DataFrame:
        self.prepare()
        assert self._cells is not None
        return self._cells

    @property
    def features(self) -> np.ndarray:
        self.prepare()
        assert self._features is not None
        return self._features

    @property
    def markers(self) -> tuple[str, ...]:
        self.prepare()
        assert self._markers is not None
        return self._markers

    @property
    def assignments(self) -> Mapping[str, pd.DataFrame]:
        self.prepare()
        return self._assignments

    @property
    def predictions(self) -> Mapping[tuple[str, str], pd.DataFrame]:
        self.prepare()
        return self._predictions

    def prepare(self) -> None:
        if self._prepared:
            return
        self._load_h5ad()
        self._validate_tiffs()
        self._run_clusterers()
        self._predictions = self._load_cached_predictions() if self.inputs.llm_mode == "cached" else self._generate_live_predictions()
        self._prepared = True

    def _load_h5ad(self) -> None:
        try:
            import anndata as ad
            from scipy import sparse
        except ImportError as exc:  # pragma: no cover
            raise SourceRebuildError("H5AD rebuilding requires anndata and scipy") from exc
        dataset = ad.read_h5ad(_require_file(self.inputs.h5ad, "H5AD source"), backed="r")
        try:
            required = {"File_ID", "ID", "x", "y", "cell_type_update"}
            missing = sorted(required.difference(dataset.obs.columns))
            if missing:
                raise SourceRebuildError(f"H5AD is missing required obs columns: {missing}")
            file_ids = dataset.obs["File_ID"].astype(str)
            positions = np.flatnonzero(file_ids.isin(COHORT_FILE_IDS).to_numpy())
            obs = dataset.obs.iloc[positions][["File_ID", "ID", "x", "y", "cell_type_update"]].copy()
            obs["File_ID"] = obs["File_ID"].astype(str)
            obs["ID"] = pd.to_numeric(obs["ID"], errors="raise").astype("int64")
            obs["x"] = pd.to_numeric(obs["x"], errors="raise").astype(float)
            obs["y"] = pd.to_numeric(obs["y"], errors="raise").astype(float)
            if obs.duplicated(["File_ID", "ID"]).any():
                raise SourceRebuildError("H5AD cohort contains duplicate exact keys")
            values = dataset.X[positions]
            if sparse.issparse(values):
                values = values.toarray()
            values = np.asarray(values, dtype=np.float32)
            markers = tuple(str(value) for value in dataset.var_names)
        finally:
            dataset.file.close()
        if len(obs) != 220_082:
            raise SourceRebuildError(f"H5AD cohort contains {len(obs):,} rows; expected 220,082")
        truth = obs["cell_type_update"].astype(str).replace(TRUTH_MAP)
        invalid = sorted(set(truth) - set(ONTOLOGY))
        if invalid:
            raise SourceRebuildError(f"H5AD truth labels are outside the ontology: {invalid}")
        obs["truth_label"] = truth.to_numpy()
        self._cells, self._features, self._markers = obs.reset_index(drop=True), values, markers

    def _validate_tiffs(self) -> None:
        assert self._cells is not None and self._markers is not None
        try:
            from llm_spatial_omics_clustering.reproduction_v2.clustering import validate_tiff_mask_correspondence
        except ImportError as exc:  # pragma: no cover
            raise SourceRebuildError("TIFF validation requires reproduction_v2 clustering") from exc
        validate_tiff_mask_correspondence(self._cells, self._markers, self.inputs.tiff_root)

    def _run_clusterers(self) -> None:
        assert self._cells is not None and self._features is not None and self._markers is not None
        try:
            from llm_spatial_omics_clustering.reproduction_v2.clustering import (
                FlowSOMSettings, LeidenSettings, PixieSettings, SpatialSortSettings,
                run_flowsom, run_leiden, run_spatialsort, run_tiff_pixie,
            )
        except ImportError as exc:  # pragma: no cover
            raise SourceRebuildError("final clustering requires reproduction_v2 clustering") from exc
        runs = [
            run_leiden(self._cells, self._features, LeidenSettings(n_neighbors=30, resolution=1.0, n_pcs=30, seed=42)),
            run_flowsom(self._cells, self._features, FlowSOMSettings(xdim=32, ydim=32, n_clusters=300, seed=42)),
            run_spatialsort(
                self._cells, self._features,
                SpatialSortSettings(n_neighbors=18, n_clusters=300, precision_scale=0.5, num_iterations=256, dmh_iterations=1, seed=42),
                output_dir=self._scratch / "spatialsort",
                source_root=Path(os.environ.get("SOURCE_REBUILD_SPATIALSORT_SOURCE_ROOT", str(ROOT / "Cluster_Comparison" / "SpatialSort"))),
                use_fast_kernel=False,
                use_fast_beta_dmh_2k=False,
            ),
            run_tiff_pixie(
                self._cells, self._markers, self._features,
                output_root=self._scratch / "pixie",
                tiffs_dir=self.inputs.tiff_root,
                runner_path=ROOT / "PIXIE" / "run_streaming_tiff_pixie.py",
                settings=PixieSettings(pixel_som_side=10, pixel_meta_clusters=20, cell_som_side=24, cell_meta_clusters=300, cell_som_sigma=2.0, cell_som_learning_rate=0.3, cell_som_iterations=5000, seed=42, include_hoechst=False),
                source_h5ad_sha256=_sha256(self.inputs.h5ad),
            ),
        ]
        for run in runs:
            key = "pixie_tiff" if run.method == "pixie" else run.method
            frame = run.assignments[["File_ID", "ID", "cluster"]].copy()
            frame["cluster"] = frame["cluster"].map(_canonical_cluster)
            self._assignments[key] = frame

    def _load_cached_predictions(self) -> dict[tuple[str, str], pd.DataFrame]:
        responses = _json_files(self.inputs.llm_data_root, "responses")
        if not responses:
            raise SourceRebuildError(f"No raw LLM response bundles found under {self.inputs.llm_data_root}")
        labels: dict[tuple[str, str], dict[str, str]] = {}
        for path in responses:
            payload = json.loads(path.read_text(encoding="utf-8"))
            if not isinstance(payload, Mapping):
                continue
            method, model = _infer_method(path, payload), _infer_model(path, payload)
            if method not in METHODS or model not in MODELS:
                continue
            target = labels.setdefault((method, model), {})
            for cluster, label in _annotation_pairs(payload):
                if cluster in target and target[cluster] != label:
                    raise SourceRebuildError(f"Conflicting cached labels for {method}/{model}/{cluster}")
                target[cluster] = label
        expected = {(method, model) for method in METHODS for model in MODELS}
        missing_sources = sorted(expected - set(labels))
        if missing_sources:
            raise SourceRebuildError(f"Raw LLM API data lacks method/model sources: {missing_sources}")
        predictions: dict[tuple[str, str], pd.DataFrame] = {}
        for method in METHODS:
            assignment = self._assignments[method]
            for model in MODELS:
                label_map = labels[(method, model)]
                missing = sorted(set(assignment["cluster"]) - set(label_map))
                if missing:
                    raise SourceRebuildError(f"Raw LLM data lacks {len(missing)} {method}/{model} cluster labels")
                frame = assignment[["File_ID", "ID"]].copy()
                frame["predicted_label"] = assignment["cluster"].map(label_map).to_numpy()
                predictions[(method, model)] = frame
        return predictions

    def _cluster_profiles(self, method: str) -> list[dict[str, Any]]:
        assert self._features is not None and self._markers is not None and self._cells is not None
        assignment = self._assignments[method]
        transformed = np.arcsinh(self._features / 5.0)
        rows: list[dict[str, Any]] = []
        for cluster, indices in assignment.groupby("cluster", sort=True).groups.items():
            values = transformed[np.asarray(indices, dtype=np.int64)]
            mean = values.mean(axis=0)
            order = np.argsort(mean)
            rows.append({"cluster": str(cluster), "n_cells": int(len(indices)), "top_markers": [self.markers[int(i)] for i in order[-12:][::-1]], "low_markers": [self.markers[int(i)] for i in order[:8]]})
        return rows

    def _generate_live_predictions(self) -> dict[tuple[str, str], pd.DataFrame]:
        key = str(self.inputs.api_keys.get("openrouter", PLACEHOLDER_API_KEY))
        if key == PLACEHOLDER_API_KEY or not key.strip():
            raise SourceRebuildError("Live mode requires OPENROUTER_API_KEY; the notebook contains only a placeholder")
        predictions: dict[tuple[str, str], pd.DataFrame] = {}
        for method in METHODS:
            profiles = self._cluster_profiles(method)
            for model in MODELS:
                labels: dict[str, str] = {}
                for start in range(0, len(profiles), 20):
                    batch = profiles[start : start + 20]
                    payload = {
                        "model": MODEL_IDS[model],
                        "temperature": 0,
                        "messages": [
                            {"role": "system", "content": "Return only a JSON object mapping cluster IDs to allowed labels."},
                            {"role": "user", "content": json.dumps({"method": method, "allowed_labels": list(ONTOLOGY), "targets": batch}, sort_keys=True)},
                        ],
                    }
                    raw = self._openrouter_request(key, payload)
                    content = raw["choices"][0]["message"]["content"]
                    parsed = json.loads(content) if isinstance(content, str) else content
                    if not isinstance(parsed, Mapping):
                        raise SourceRebuildError(f"Live {method}/{model} response was not a JSON object")
                    for cluster, label in parsed.items():
                        if str(label) not in ONTOLOGY:
                            raise SourceRebuildError(f"Live response used an invalid ontology label: {label}")
                        labels[_canonical_cluster(cluster)] = str(label)
                    if self.inputs.live_output_root is not None:
                        output = self.inputs.live_output_root / method / model / f"batch_{start // 20:03d}.json"
                        output.parent.mkdir(parents=True, exist_ok=True)
                        output.write_text(json.dumps({"method": method, "model": model, "request": payload, "raw_response": raw, "annotations": parsed}, indent=2) + "\n", encoding="utf-8")
                assignment = self._assignments[method]
                missing = sorted(set(assignment["cluster"]) - set(labels))
                if missing:
                    raise SourceRebuildError(f"Live {method}/{model} response missed clusters: {missing[:5]}")
                frame = assignment[["File_ID", "ID"]].copy()
                frame["predicted_label"] = assignment["cluster"].map(labels).to_numpy()
                predictions[(method, model)] = frame
        return predictions

    @staticmethod
    def _openrouter_request(api_key: str, payload: Mapping[str, Any]) -> Mapping[str, Any]:
        request = urllib_request.Request("https://openrouter.ai/api/v1/chat/completions", data=json.dumps(payload).encode("utf-8"), headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json", "HTTP-Referer": "https://localhost/", "X-Title": "Cell masks final source rebuild"}, method="POST")
        with urllib_request.urlopen(request, timeout=300) as response:
            result = json.loads(response.read().decode("utf-8"))
        if not isinstance(result, Mapping):
            raise SourceRebuildError("OpenRouter returned a non-object response")
        return result

    def embedding(self) -> np.ndarray:
        self.prepare()
        if self._embedding is None:
            from sklearn.decomposition import PCA

            self._embedding = PCA(n_components=2, random_state=42, svd_solver="randomized").fit_transform(np.arcsinh(self.features / 5.0))
        return self._embedding

    def metric_frame(self, method: str, model: str) -> pd.DataFrame:
        frame = self.cells[["File_ID", "ID", "truth_label"]].merge(self.predictions[(method, model)], on=["File_ID", "ID"], validate="one_to_one")
        frame["correct"] = frame["truth_label"].eq(frame["predicted_label"])
        frame["cluster"] = self._assignments[method]["cluster"].to_numpy()
        return frame

    def accuracy_matrix(self) -> pd.DataFrame:
        return pd.DataFrame({model: [float(self.metric_frame(method, model)["correct"].mean()) for method in METHODS] for model in MODELS}, index=METHODS)

    def truth_label_order(self) -> list[str]:
        return list(self.cells["truth_label"].value_counts().index.astype(str))


def source_context_from_environment() -> SourceContext:
    return SourceContext(SourceInputs.from_environment())


def cluster_configuration_frame() -> pd.DataFrame:
    return pd.DataFrame([
        {"Method": "Leiden", "Configured clusters": 300, "Settings": "PCA 30; neighbors 30; resolution 1.0; seed 42"},
        {"Method": "FlowSOM", "Configured clusters": 300, "Settings": "MiniSOM 32x32; seed 42"},
        {"Method": "SpatialSort", "Configured clusters": 300, "Settings": "neighbors 18; precision 0.5; 256 iterations; DMH 1; seed 42"},
        {"Method": "TIFF PIXIE", "Configured clusters": 300, "Settings": "pixel SOM 10x10; pixel meta 20; cell SOM 24x24; seed 42"},
    ])


def runtime_manifest() -> dict[str, Any]:
    return {
        "schema_version": "cell_masks.final_source_rebuild_runtime.v1",
        "runtime_dependency_boundary": "H5AD + paired OME-TIFF files + OpenRouter key (or cached raw LLM API bundles)",
        "runtime_helper": "source_rebuild_scripts/final_source_rebuild_runtime.py",
        "panel_renderer": "source_rebuild_scripts/final_source_rebuild_panels.py",
        "allowed_input_environment": ["SOURCE_REBUILD_REPO_ROOT", "SOURCE_REBUILD_H5AD_PATH", "SOURCE_REBUILD_TIFF_ROOT", "SOURCE_REBUILD_LLM_DATA_ROOT", "SOURCE_REBUILD_LLM_MODE", "SOURCE_REBUILD_OUTPUT_ROOT", "SOURCE_REBUILD_LLM_OUTPUT_ROOT"],
        "api_key_environment": "OPENROUTER_API_KEY",
        "api_key_policy": "placeholder in notebook; environment-only in live mode; never persisted",
        "cohort_file_ids": list(COHORT_FILE_IDS),
        "methods": list(METHODS),
        "models": list(MODELS),
    }


## 3. Recompute clustering and LLM annotations in memory


In [ ]:
# Build all source-derived state once for this notebook process.
# This loads the H5AD, validates TIFF/mask pairs, recomputes every clustering method,
# calls OpenRouter in live mode, and retains assignments/annotations only in memory.
SOURCE_INPUTS = SourceInputs.from_environment()
SOURCE_CONTEXT = SourceContext(SOURCE_INPUTS)
SOURCE_CONTEXT.prepare()
PANEL_FIGURES = {}
display(pd.DataFrame([SOURCE_CONTEXT.status()]))


## 4. Render publication panels


### Panel A

vector workflow rebuilt in Matplotlib.


In [ ]:
# Panel A: vector workflow rebuilt in Matplotlib

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def _workflow_box(ax, xy, width, height, text, facecolor, *, fontsize=9):
    box = FancyBboxPatch(
        xy,
        width,
        height,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        linewidth=1.2,
        edgecolor="#1f2933",
        facecolor=facecolor,
    )
    ax.add_patch(box)
    ax.text(
        xy[0] + width / 2,
        xy[1] + height / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        color="#101820",
        wrap=True,
    )

def _workflow_arrow(ax, start, end):
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=12,
            linewidth=1.2,
            color="#52606d",
            connectionstyle="arc3,rad=0.0",
        )
    )

def render_figure_03_panel_a(context):
    """Rebuild the Figure 3A LLM annotation workflow as vector art."""
    del context
    fig, ax = plt.subplots(figsize=(12, 4.8), constrained_layout=True)
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 4.8)
    ax.axis("off")
    ax.text(0.05, 4.45, "A", fontsize=16, fontweight="bold", va="center")
    ax.text(0.55, 4.45, "LLM-assisted cell-type annotation benchmark", fontsize=14, fontweight="bold", va="center")

    _workflow_box(ax, (0.45, 2.45), 1.7, 1.05, "Marker\nprofiles", "#d9eef7")
    _workflow_box(ax, (2.9, 2.45), 1.7, 1.05, "Cluster\nsummary", "#d9eef7")
    _workflow_box(ax, (5.35, 3.25), 1.8, 0.82, "Four methods", "#f9e3b8")
    _workflow_box(ax, (5.35, 2.05), 1.8, 0.82, "Four models", "#f9e3b8")
    _workflow_box(ax, (5.35, 0.85), 1.8, 0.82, "Raw responses", "#f9e3b8")
    _workflow_box(ax, (8.0, 2.45), 1.9, 1.05, "Cell-level\npredictions", "#d9eef7")
    _workflow_box(ax, (10.55, 2.45), 1.0, 1.05, "F1", "#cdebd6")
    _workflow_arrow(ax, (2.15, 2.98), (2.9, 2.98))
    _workflow_arrow(ax, (4.6, 2.98), (5.35, 3.65))
    _workflow_arrow(ax, (4.6, 2.98), (5.35, 2.46))
    _workflow_arrow(ax, (4.6, 2.98), (5.35, 1.26))
    _workflow_arrow(ax, (7.15, 3.65), (8.0, 2.98))
    _workflow_arrow(ax, (7.15, 2.46), (8.0, 2.98))
    _workflow_arrow(ax, (7.15, 1.26), (8.0, 2.98))
    _workflow_arrow(ax, (9.9, 2.98), (10.55, 2.98))
    ax.text(6.0, 0.27, "Cached raw API bundles are the default; live regeneration is opt-in and environment-keyed", ha="center", fontsize=9, color="#52606d")
    return fig

FIGURE = render_figure_03_panel_a(SOURCE_CONTEXT)
PANEL_FIGURES['A'] = FIGURE
display(FIGURE)

### Panel B

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel B: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _llm_metrics(context: SourceContext) -> pd.DataFrame:
    truth = _truth(context)
    labels = sorted(truth.unique())
    records: list[dict[str, Any]] = []
    for method in METHODS:
        for model in MODELS:
            predicted = _prediction(context, method, model)
            records.append(
                {
                    "method": method,
                    "method_label": METHOD_LABELS[method],
                    "model": model,
                    "model_label": MODEL_LABELS[model],
                    "accuracy": float((truth == predicted).mean()),
                    "macro_f1": float(f1_score(truth, predicted, labels=labels, average="macro", zero_division=0)),
                    "weighted_f1": float(f1_score(truth, predicted, labels=labels, average="weighted", zero_division=0)),
                }
            )
    return pd.DataFrame.from_records(records)

def _heatmap(axis: Any, matrix: pd.DataFrame, title: str, *, vmin: float | None = None, vmax: float | None = None, cmap: str = "viridis") -> Any:
    image = axis.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set(
        title=title,
        xticks=np.arange(matrix.shape[1]),
        xticklabels=list(matrix.columns),
        yticks=np.arange(matrix.shape[0]),
        yticklabels=list(matrix.index),
    )
    axis.tick_params(axis="x", rotation=45)
    if matrix.shape[0] * matrix.shape[1] <= 64:
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                axis.text(column_index, row_index, f"{matrix.iat[row_index, column_index]:.2f}", ha="center", va="center", fontsize=7)
    return image

def render_figure_03_panel_b(context: SourceContext) -> Any:
    frame = _llm_metrics(context)
    matrix = frame.pivot(index="method_label", columns="model_label", values="macro_f1")
    matrix = matrix.reindex(index=[METHOD_LABELS[m] for m in METHODS], columns=[MODEL_LABELS[m] for m in MODELS])
    figure, axis = plt.subplots(figsize=(7.0, 4.8))
    image = _heatmap(axis, matrix, "LLM macro F1 by clustering method and model", vmin=0, vmax=1)
    figure.colorbar(image, ax=axis, label="Macro F1")
    _add_panel_label(figure, "B")
    figure.tight_layout()
    return figure

FIGURE = render_figure_03_panel_b(SOURCE_CONTEXT)
PANEL_FIGURES['B'] = FIGURE
display(FIGURE)

### Panel C

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel C: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _llm_metrics(context: SourceContext) -> pd.DataFrame:
    truth = _truth(context)
    labels = sorted(truth.unique())
    records: list[dict[str, Any]] = []
    for method in METHODS:
        for model in MODELS:
            predicted = _prediction(context, method, model)
            records.append(
                {
                    "method": method,
                    "method_label": METHOD_LABELS[method],
                    "model": model,
                    "model_label": MODEL_LABELS[model],
                    "accuracy": float((truth == predicted).mean()),
                    "macro_f1": float(f1_score(truth, predicted, labels=labels, average="macro", zero_division=0)),
                    "weighted_f1": float(f1_score(truth, predicted, labels=labels, average="weighted", zero_division=0)),
                }
            )
    return pd.DataFrame.from_records(records)

def render_figure_03_panel_c(context: SourceContext) -> Any:
    frame = _llm_metrics(context)
    figure, axis = plt.subplots(figsize=(7.2, 4.6))
    positions = np.arange(len(METHODS))
    for model_index, model in enumerate(MODELS):
        subset = frame.loc[frame["model"].eq(model)].set_index("method").reindex(METHODS)
        axis.plot(positions, subset["accuracy"], marker="o", linewidth=1.8, label=MODEL_LABELS[model])
    axis.set(xticks=positions, xticklabels=[METHOD_LABELS[m] for m in METHODS], ylim=(0, 1), ylabel="Cell-level accuracy", title="LLM accuracy across clustering methods")
    axis.tick_params(axis="x", rotation=35)
    axis.legend(frameon=False, ncols=2)
    _add_panel_label(figure, "C")
    figure.tight_layout()
    return figure

FIGURE = render_figure_03_panel_c(SOURCE_CONTEXT)
PANEL_FIGURES['C'] = FIGURE
display(FIGURE)

### Panel D

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel D: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _llm_metrics(context: SourceContext) -> pd.DataFrame:
    truth = _truth(context)
    labels = sorted(truth.unique())
    records: list[dict[str, Any]] = []
    for method in METHODS:
        for model in MODELS:
            predicted = _prediction(context, method, model)
            records.append(
                {
                    "method": method,
                    "method_label": METHOD_LABELS[method],
                    "model": model,
                    "model_label": MODEL_LABELS[model],
                    "accuracy": float((truth == predicted).mean()),
                    "macro_f1": float(f1_score(truth, predicted, labels=labels, average="macro", zero_division=0)),
                    "weighted_f1": float(f1_score(truth, predicted, labels=labels, average="weighted", zero_division=0)),
                }
            )
    return pd.DataFrame.from_records(records)

def _heatmap(axis: Any, matrix: pd.DataFrame, title: str, *, vmin: float | None = None, vmax: float | None = None, cmap: str = "viridis") -> Any:
    image = axis.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set(
        title=title,
        xticks=np.arange(matrix.shape[1]),
        xticklabels=list(matrix.columns),
        yticks=np.arange(matrix.shape[0]),
        yticklabels=list(matrix.index),
    )
    axis.tick_params(axis="x", rotation=45)
    if matrix.shape[0] * matrix.shape[1] <= 64:
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                axis.text(column_index, row_index, f"{matrix.iat[row_index, column_index]:.2f}", ha="center", va="center", fontsize=7)
    return image

def render_figure_03_panel_d(context: SourceContext) -> Any:
    frame = _llm_metrics(context)
    matrix = frame.pivot(index="method_label", columns="model_label", values="weighted_f1")
    matrix = matrix.reindex(index=[METHOD_LABELS[m] for m in METHODS], columns=[MODEL_LABELS[m] for m in MODELS])
    figure, axis = plt.subplots(figsize=(7.0, 4.8))
    image = _heatmap(axis, matrix, "LLM weighted F1 by clustering method and model", vmin=0, vmax=1)
    figure.colorbar(image, ax=axis, label="Weighted F1")
    _add_panel_label(figure, "D")
    figure.tight_layout()
    return figure

FIGURE = render_figure_03_panel_d(SOURCE_CONTEXT)
PANEL_FIGURES['D'] = FIGURE
display(FIGURE)

### Panel E

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel E: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _llm_metrics(context: SourceContext) -> pd.DataFrame:
    truth = _truth(context)
    labels = sorted(truth.unique())
    records: list[dict[str, Any]] = []
    for method in METHODS:
        for model in MODELS:
            predicted = _prediction(context, method, model)
            records.append(
                {
                    "method": method,
                    "method_label": METHOD_LABELS[method],
                    "model": model,
                    "model_label": MODEL_LABELS[model],
                    "accuracy": float((truth == predicted).mean()),
                    "macro_f1": float(f1_score(truth, predicted, labels=labels, average="macro", zero_division=0)),
                    "weighted_f1": float(f1_score(truth, predicted, labels=labels, average="weighted", zero_division=0)),
                }
            )
    return pd.DataFrame.from_records(records)

def render_figure_03_panel_e(context: SourceContext) -> Any:
    frame = _llm_metrics(context)
    summary = frame.groupby("model_label", sort=False)["macro_f1"].agg(["mean", "std"]).reindex([MODEL_LABELS[m] for m in MODELS])
    figure, axis = plt.subplots(figsize=(6.5, 4.4))
    axis.bar(summary.index, summary["mean"], yerr=summary["std"], color="#756bb1", capsize=3)
    axis.set(ylim=(0, 1), ylabel="Macro F1", title="Average LLM annotation performance")
    _add_panel_label(figure, "E")
    figure.tight_layout()
    return figure

FIGURE = render_figure_03_panel_e(SOURCE_CONTEXT)
PANEL_FIGURES['E'] = FIGURE
display(FIGURE)

### Panel F

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel F: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _per_type_f1(context: SourceContext, method: str, model: str, n_types: int = 12) -> pd.Series:
    truth = _truth(context)
    predicted = _prediction(context, method, model)
    types = truth.value_counts().head(n_types).index.astype(str)
    return pd.Series(
        {
            label: float(
                f1_score(
                    truth.eq(label),
                    predicted.eq(label),
                    zero_division=0,
                )
            )
            for label in types
        },
        name=f"{method}/{model}",
    )

def _heatmap(axis: Any, matrix: pd.DataFrame, title: str, *, vmin: float | None = None, vmax: float | None = None, cmap: str = "viridis") -> Any:
    image = axis.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set(
        title=title,
        xticks=np.arange(matrix.shape[1]),
        xticklabels=list(matrix.columns),
        yticks=np.arange(matrix.shape[0]),
        yticklabels=list(matrix.index),
    )
    axis.tick_params(axis="x", rotation=45)
    if matrix.shape[0] * matrix.shape[1] <= 64:
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                axis.text(column_index, row_index, f"{matrix.iat[row_index, column_index]:.2f}", ha="center", va="center", fontsize=7)
    return image

def render_figure_03_panel_f(context: SourceContext) -> Any:
    matrix = pd.DataFrame({_method: _per_type_f1(context, _method, "gpt") for _method in METHODS})
    matrix.columns = [METHOD_LABELS[m] for m in METHODS]
    figure, axis = plt.subplots(figsize=(7.5, 6.0))
    image = _heatmap(axis, matrix, "GPT per-cell-type F1", vmin=0, vmax=1)
    figure.colorbar(image, ax=axis, label="F1")
    _add_panel_label(figure, "F")
    figure.tight_layout()
    return figure

FIGURE = render_figure_03_panel_f(SOURCE_CONTEXT)
PANEL_FIGURES['F'] = FIGURE
display(FIGURE)

### Panel G

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel G: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _llm_metrics(context: SourceContext) -> pd.DataFrame:
    truth = _truth(context)
    labels = sorted(truth.unique())
    records: list[dict[str, Any]] = []
    for method in METHODS:
        for model in MODELS:
            predicted = _prediction(context, method, model)
            records.append(
                {
                    "method": method,
                    "method_label": METHOD_LABELS[method],
                    "model": model,
                    "model_label": MODEL_LABELS[model],
                    "accuracy": float((truth == predicted).mean()),
                    "macro_f1": float(f1_score(truth, predicted, labels=labels, average="macro", zero_division=0)),
                    "weighted_f1": float(f1_score(truth, predicted, labels=labels, average="weighted", zero_division=0)),
                }
            )
    return pd.DataFrame.from_records(records)

def render_figure_03_panel_g(context: SourceContext) -> Any:
    frame = _llm_metrics(context)
    figure, axis = plt.subplots(figsize=(8.0, 4.8))
    positions = np.arange(len(METHODS))
    width = 0.18
    for model_index, model in enumerate(MODELS):
        subset = frame.loc[frame["model"].eq(model)].set_index("method").reindex(METHODS)
        axis.bar(positions + (model_index - 1.5) * width, subset["macro_f1"], width=width, label=MODEL_LABELS[model])
    axis.set(xticks=positions, xticklabels=[METHOD_LABELS[m] for m in METHODS], ylim=(0, 1), ylabel="Macro F1", title="Model comparison within each clustering method")
    axis.tick_params(axis="x", rotation=35)
    axis.legend(frameon=False, ncols=2)
    _add_panel_label(figure, "G")
    figure.tight_layout()
    return figure

FIGURE = render_figure_03_panel_g(SOURCE_CONTEXT)
PANEL_FIGURES['G'] = FIGURE
display(FIGURE)

### Panel H

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel H: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _per_type_f1(context: SourceContext, method: str, model: str, n_types: int = 12) -> pd.Series:
    truth = _truth(context)
    predicted = _prediction(context, method, model)
    types = truth.value_counts().head(n_types).index.astype(str)
    return pd.Series(
        {
            label: float(
                f1_score(
                    truth.eq(label),
                    predicted.eq(label),
                    zero_division=0,
                )
            )
            for label in types
        },
        name=f"{method}/{model}",
    )

def _heatmap(axis: Any, matrix: pd.DataFrame, title: str, *, vmin: float | None = None, vmax: float | None = None, cmap: str = "viridis") -> Any:
    image = axis.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set(
        title=title,
        xticks=np.arange(matrix.shape[1]),
        xticklabels=list(matrix.columns),
        yticks=np.arange(matrix.shape[0]),
        yticklabels=list(matrix.index),
    )
    axis.tick_params(axis="x", rotation=45)
    if matrix.shape[0] * matrix.shape[1] <= 64:
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                axis.text(column_index, row_index, f"{matrix.iat[row_index, column_index]:.2f}", ha="center", va="center", fontsize=7)
    return image

def render_figure_03_panel_h(context: SourceContext) -> Any:
    matrices = [_per_type_f1(context, method, model) for method in METHODS for model in MODELS]
    frame = pd.concat(matrices, axis=1)
    frame.columns = [f"{METHOD_LABELS[method]}\n{MODEL_LABELS[model]}" for method in METHODS for model in MODELS]
    figure, axis = plt.subplots(figsize=(12.0, 6.0))
    image = _heatmap(axis, frame, "Cell-type F1 across all LLM and clustering combinations", vmin=0, vmax=1)
    figure.colorbar(image, ax=axis, label="F1")
    _add_panel_label(figure, "H")
    figure.tight_layout()
    return figure

FIGURE = render_figure_03_panel_h(SOURCE_CONTEXT)
PANEL_FIGURES['H'] = FIGURE
display(FIGURE)

### Panel I

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel I: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _heatmap(axis: Any, matrix: pd.DataFrame, title: str, *, vmin: float | None = None, vmax: float | None = None, cmap: str = "viridis") -> Any:
    image = axis.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set(
        title=title,
        xticks=np.arange(matrix.shape[1]),
        xticklabels=list(matrix.columns),
        yticks=np.arange(matrix.shape[0]),
        yticklabels=list(matrix.index),
    )
    axis.tick_params(axis="x", rotation=45)
    if matrix.shape[0] * matrix.shape[1] <= 64:
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                axis.text(column_index, row_index, f"{matrix.iat[row_index, column_index]:.2f}", ha="center", va="center", fontsize=7)
    return image

def render_figure_03_panel_i(context: SourceContext) -> Any:
    method = "leiden"
    predictions = {model: _prediction(context, method, model) for model in MODELS}
    matrix = pd.DataFrame(
        {
            left: {right: float((predictions[left] == predictions[right]).mean()) for right in MODELS}
            for left in MODELS
        }
    ).rename(index=MODEL_LABELS, columns=MODEL_LABELS)
    figure, axis = plt.subplots(figsize=(5.6, 4.8))
    image = _heatmap(axis, matrix, "Leiden label agreement between LLMs", vmin=0, vmax=1)
    figure.colorbar(image, ax=axis, label="Cell-level agreement")
    _add_panel_label(figure, "I")
    figure.tight_layout()
    return figure

FIGURE = render_figure_03_panel_i(SOURCE_CONTEXT)
PANEL_FIGURES['I'] = FIGURE
display(FIGURE)

### Panel J

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel J: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _palette(values: pd.Series) -> tuple[np.ndarray, Any]:
    labels = pd.Index(values.astype(str).unique()).sort_values()
    categories = pd.Categorical(values.astype(str), categories=labels, ordered=True)
    return categories.codes, plt.get_cmap("turbo", max(1, len(labels)))

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _representative_fov(context: SourceContext) -> str:
    counts = _prepare(context).cells["File_ID"].astype(str).value_counts()
    return str(counts.sort_values(ascending=False).index[0])

def _spatial_scatter(axis: Any, context: SourceContext, labels: pd.Series, title: str) -> None:
    cells = _prepare(context).cells
    fov = _representative_fov(context)
    mask = cells["File_ID"].astype(str).eq(fov).to_numpy()
    codes, cmap = _palette(labels.loc[mask].reset_index(drop=True))
    axis.scatter(cells.loc[mask, "x"], cells.loc[mask, "y"], c=codes, cmap=cmap, s=0.45, linewidths=0, rasterized=True)
    axis.set(title=title, xticks=[], yticks=[], aspect="equal")
    axis.invert_yaxis()

def render_figure_03_panel_j(context: SourceContext) -> Any:
    figure, axes = plt.subplots(1, 2, figsize=(10.0, 4.8), constrained_layout=True)
    _spatial_scatter(axes[0], context, _truth(context), "H5AD truth labels")
    _spatial_scatter(axes[1], context, _prediction(context, "leiden", "gpt"), "Leiden plus GPT labels")
    _add_panel_label(figure, "J")
    return figure

FIGURE = render_figure_03_panel_j(SOURCE_CONTEXT)
PANEL_FIGURES['J'] = FIGURE
display(FIGURE)

### Panel K

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel K: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _heatmap(axis: Any, matrix: pd.DataFrame, title: str, *, vmin: float | None = None, vmax: float | None = None, cmap: str = "viridis") -> Any:
    image = axis.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set(
        title=title,
        xticks=np.arange(matrix.shape[1]),
        xticklabels=list(matrix.columns),
        yticks=np.arange(matrix.shape[0]),
        yticklabels=list(matrix.index),
    )
    axis.tick_params(axis="x", rotation=45)
    if matrix.shape[0] * matrix.shape[1] <= 64:
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                axis.text(column_index, row_index, f"{matrix.iat[row_index, column_index]:.2f}", ha="center", va="center", fontsize=7)
    return image

def render_figure_03_panel_k(context: SourceContext) -> Any:
    truth = _truth(context)
    predicted = _prediction(context, "leiden", "gpt")
    types = truth.value_counts().head(12).index.astype(str)
    matrix = pd.crosstab(truth, predicted).reindex(index=types, columns=types, fill_value=0)
    matrix = matrix.div(matrix.sum(axis=1).replace(0, 1), axis=0)
    figure, axis = plt.subplots(figsize=(7.2, 6.2))
    image = _heatmap(axis, matrix, "Leiden plus GPT normalized confusion matrix", vmin=0, vmax=1, cmap="Blues")
    figure.colorbar(image, ax=axis, label="Fraction of truth cell type")
    _add_panel_label(figure, "K")
    figure.tight_layout()
    return figure

FIGURE = render_figure_03_panel_k(SOURCE_CONTEXT)
PANEL_FIGURES['K'] = FIGURE
display(FIGURE)

### Panel L

H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering.


In [ ]:
# Panel L: H5AD/TIFF clustering, live/cached LLM annotation, and Matplotlib rendering

# The code below is embedded in this notebook cell; it does not import a panel renderer or image.

from collections.abc import Mapping

from typing import Any

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from sklearn.metrics import f1_score

def _prepare(context: SourceContext) -> SourceContext:
    context.prepare()
    return context

def _keys(context: SourceContext) -> pd.DataFrame:
    return _prepare(context).cells.loc[:, ["File_ID", "ID"]].copy()

def _truth(context: SourceContext) -> pd.Series:
    return _prepare(context).cells["truth_label"].astype(str).reset_index(drop=True)

def _prediction(context: SourceContext, method: str, model: str) -> pd.Series:
    key = (method, model)
    if key not in _prepare(context).predictions:
        raise SourceRebuildError(f"No LLM annotations are available for {method}/{model}")
    frame = context.predictions[key].loc[:, ["File_ID", "ID", "predicted_label"]]
    merged = _keys(context).merge(frame, on=["File_ID", "ID"], how="left", validate="one_to_one", sort=False)
    if merged["predicted_label"].isna().any():
        raise SourceRebuildError(f"{method}/{model} did not annotate every source cell")
    return merged["predicted_label"].astype(str).reset_index(drop=True)

def _palette(values: pd.Series) -> tuple[np.ndarray, Any]:
    labels = pd.Index(values.astype(str).unique()).sort_values()
    categories = pd.Categorical(values.astype(str), categories=labels, ordered=True)
    return categories.codes, plt.get_cmap("turbo", max(1, len(labels)))

def _add_panel_label(figure: Any, panel_id: str) -> None:
    figure.text(0.01, 0.99, panel_id, ha="left", va="top", fontsize=18, fontweight="bold")

def _umap_scatter(axis: Any, context: SourceContext, labels: pd.Series, title: str) -> None:
    coordinates = _prepare(context).embedding()
    codes, cmap = _palette(labels)
    axis.scatter(coordinates[:, 0], coordinates[:, 1], c=codes, cmap=cmap, s=0.28, linewidths=0, rasterized=True)
    axis.set(title=title, xticks=[], yticks=[], xlabel="UMAP 1", ylabel="UMAP 2")

def render_figure_03_panel_l(context: SourceContext) -> Any:
    figure, axes = plt.subplots(1, 2, figsize=(10.0, 4.8), constrained_layout=True)
    _umap_scatter(axes[0], context, _truth(context), "H5AD truth labels")
    _umap_scatter(axes[1], context, _prediction(context, "leiden", "gpt"), "Leiden plus GPT labels")
    _add_panel_label(figure, "L")
    return figure

FIGURE = render_figure_03_panel_l(SOURCE_CONTEXT)
PANEL_FIGURES['L'] = FIGURE
display(FIGURE)

## 5. Export the source-rebuilt figure PDF


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

PDF_ROOT = Path(os.environ["SOURCE_REBUILD_OUTPUT_ROOT"]).expanduser().resolve()
PDF_ROOT.mkdir(parents=True, exist_ok=True)
PDF_PATH = PDF_ROOT / 'Figure_03_source_rebuilt.pdf'
with PdfPages(PDF_PATH) as pdf:
    for panel_id in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L']:
        PANEL_FIGURES[panel_id].savefig(pdf, format="pdf", bbox_inches="tight")
print(f"Wrote {PDF_PATH}")
